# Koshkina & Elder jersey-number-pipeline — SoccerNet reproduction (Colab T4)

Reproduces the tracklet-level jersey-number accuracy reported in Koshkina & Elder,
*"A General Framework for Jersey Number Recognition in Sports Video"* (CVPRW'24):
**87.45% tracklet accuracy on the SoccerNet jersey-2023 test split**, using the authors'
own repo (https://github.com/mkoshkina/jersey-number-pipeline), their own published
weights (gdown from their Google Drive IDs baked into `configuration.py`), and their own
`setup.py` / `main.py` entry points — unmodified.

- **Runtime:** GPU (T4) required. `Runtime -> Change runtime type -> T4 GPU`.
- **Expected wall time:** ~40-70 min — conda env creation + weight downloads (~15-20 min),
  SoccerNet test-split download (~5-10 min, ~2500 tracklets), full pipeline inference
  (~20-40 min on a T4).
- **Independent of the local laptop run.** The local repro
  (`~/jersey-number-pipeline/repro_soccernet.py` + `repro_str.py`) swaps ViTPose for
  torchvision KeypointRCNN because mmcv/mmpose does not build on Windows/py3.14, and
  measured **0.42** tracklet accuracy. This notebook runs the *unmodified* upstream
  pipeline (real ViTPose, real conda-isolated PARSeq/Centroid-ReID envs) on a clean Colab
  VM, so its number is a genuine cross-check of the 0.42 vs 0.8745 gap — not the same code
  path.
- **Zero uploads needed.** The repo, all model weights, and the SoccerNet test split are
  all fetched fresh inside this notebook.


In [ ]:
# GPU check
!nvidia-smi
import torch
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU visible - set Runtime > Change runtime type > T4 GPU"


## 1. Install conda (Colab has no conda by default)

The pipeline's own `setup.py` creates three **isolated conda environments**
(`vitpose` py3.8, `parseq2` py3.9, `centroids` py3.8) each pinned to the old
torch/cuda build its sub-repo (ViTPose/PARSeq/Centroid-ReID) needs — that isolation is
exactly why their old pins don't fight Colab's own (newer) base torch. `condacolab` is the
standard way to get `conda`/`mamba` on a Colab VM.

**This cell restarts the Colab runtime automatically** (condacolab requirement). After the
restart, continue running the cells below in order — `Runtime > Run all` handles the
restart correctly on its own; you do not need to intervene.

In [ ]:
import sys
!{sys.executable} -m pip install -q condacolab
import condacolab
condacolab.install()


In [ ]:
# Runs after the automatic restart above. Confirms conda is usable.
import condacolab
condacolab.check()


## 2. Clone the pipeline and run their `setup.py`

`setup.py SoccerNet` clones ViTPose, PARSeq and Centroid-ReID, creates their conda envs,
and `gdown`s the three published weights (legibility classifier, ViTPose-h pose model,
SoccerNet-fine-tuned PARSeq STR) using the Google Drive file IDs already baked into
`configuration.py` (`legibility_model_url`, `pose_model_url`, `str_model_url`).

One addition on top of a bare `setup.py` call: the repo's README states
**Requirements: pytorch 1.9.0** as a base assumption, but `setup_pose()` in `setup.py`
`pip install`s `mmcv-full==1.4.8` (built against `torch1.9.0+cu111`) into the fresh
`vitpose` conda env *without first installing that torch build* — mmcv-full's wheel needs
a matching torch already importable to link against. We pre-seed exactly that env with the
README's stated torch version before calling `setup.py`, then let `setup.py` do everything
else unmodified.

In [ ]:
%cd /content
!git clone --recurse-submodules https://github.com/mkoshkina/jersey-number-pipeline.git
%cd /content/jersey-number-pipeline

# Base driver env needs these (legibility classifier + helpers run inline, not via `conda run`).
import sys
!{sys.executable} -m pip install -q gdown opencv-python pandas scipy tqdm

# Pre-seed the vitpose env with the torch build mmcv-full==1.4.8 needs (see markdown above).
!conda create -n vitpose python=3.8 -y
!conda run -n vitpose pip install torch==1.9.0+cu111 torchvision==0.10.0+cu111 \
    -f https://download.pytorch.org/whl/torch_stable.html

!python setup.py SoccerNet


## 3. Download the SoccerNet jersey-2023 TEST split

Images (`test.zip`) are public (per-task OwnCloud key, not the gated broadcast-video NDA); labels (`test_labels.zip`) require the SoccerNet NDA password. Downloads land in the path `configuration.py` already expects: `data/SoccerNet/test/images/<tracklet>/*.jpg` and `data/SoccerNet/test/test_gt.json`.

Labels are NDA-gated: `test.zip` (images) is public, but `test_labels.zip` returns HTTP 401 without SoccerNet NDA credentials. You'll be prompted for the SoccerNet password below. Fallback if you don't have it: upload `test_gt.json` from the local repo path `data/soccernet/jersey-2023/test/test_gt.json` via `google.colab.files.upload` (commented-out snippet at the bottom of the next cell).

In [ ]:
import sys
!{sys.executable} -m pip install -q SoccerNet

import zipfile
from getpass import getpass
from pathlib import Path
from SoccerNet.Downloader import SoccerNetDownloader

DL_DIR = Path("/content/sn_download")
DL_DIR.mkdir(parents=True, exist_ok=True)

dl = SoccerNetDownloader(LocalDirectory=str(DL_DIR))

# test.zip (images) is public - skip if already downloaded.
test_zip = DL_DIR / "jersey-2023" / "test.zip"
if not test_zip.exists():
    dl.downloadDataTask(task="jersey-2023", split=["test"])

# test_labels.zip is NDA-gated - password required.
dl.password = getpass("SoccerNet NDA password: ")
dl.downloadDataTask(task="jersey-2023", split=["test_labels"])

# Land it where configuration.py's default root_dir ('./data/SoccerNet') expects it.
DEST = Path("/content/jersey-number-pipeline/data/SoccerNet")
DEST.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(test_zip) as zf:
    zf.extractall(DEST)  # SoccerNet zip layout already nests under 'test/images/...'

# Locate test_gt.json robustly: bare json next to the zips, or inside test_labels.zip.
jersey_dir = DL_DIR / "jersey-2023"
gt_src = jersey_dir / "test_gt.json"
if not gt_src.exists():
    labels_zip = jersey_dir / "test_labels.zip"
    with zipfile.ZipFile(labels_zip) as zf:
        zf.extractall(jersey_dir)
    gt_src = next(jersey_dir.rglob("test_gt.json"))

gt_dst = DEST / "test" / "test_gt.json"
gt_dst.parent.mkdir(parents=True, exist_ok=True)
gt_dst.write_bytes(gt_src.read_bytes())

n_tracklets = len(list((DEST / "test" / "images").iterdir()))
print(f"test tracklets: {n_tracklets}, gt file: {gt_dst.exists()}")

# Fallback if you don't have the SoccerNet NDA password: comment out the
# dl.password / downloadDataTask(test_labels) lines above and upload test_gt.json
# from the local repo instead:
#
# from google.colab import files
# uploaded = files.upload()  # select data/soccernet/jersey-2023/test/test_gt.json
# gt_dst.parent.mkdir(parents=True, exist_ok=True)
# Path("test_gt.json").rename(gt_dst)


## 4. Run the full pipeline (their own entry point)

`main.py SoccerNet test` runs their real driver end to end: soccer-ball filter -> ReID
feature extraction + Gaussian outlier removal -> legibility classification -> ViTPose
pose -> pose-guided crops -> PARSeq STR -> tracklet-vote consolidation -> their own eval
print.

**Re-run note:** `main.py`'s stages are not individually checkpointed — re-running this
cell restarts the whole pipeline from the soccer-ball filter. If a Colab disconnect
interrupts a run partway through, edit the `actions` dict at the bottom of
`jersey-number-pipeline/main.py` (set already-completed stages to `False`) before
re-running, rather than starting over.

In [ ]:
%cd /content/jersey-number-pipeline
!python main.py SoccerNet test


## 5. Score: predictions vs `test_gt.json`

`main.py`'s own `eval` stage already prints their headline number (their
`SKIP_ILLEGIBLE` convention: tracklets either side calls illegible/-1 are dropped from
both numerator and denominator). This cell recomputes it independently from the saved
JSON files, plus a stricter **numbered-only** accuracy (denominator = tracklets with a
real ground-truth number; a predicted `-1` on those counts as wrong), and prints both next
to the paper's claimed 87.45% and the local Windows-repro baseline of 0.42.

In [ ]:
import json
from pathlib import Path

WORK = Path("/content/jersey-number-pipeline/out/SoccerNetResults")
final = json.loads((WORK / "final_results.json").read_text())
gt = json.loads(
    Path("/content/jersey-number-pipeline/data/SoccerNet/test/test_gt.json").read_text()
)


def overall_accuracy(final, gt):
    """Their SKIP_ILLEGIBLE metric: drop tracklets illegible on either side."""
    correct = total = 0
    for tid, g in gt.items():
        p = final.get(tid, -1)
        if g == -1 or p == -1:
            continue
        total += 1
        correct += int(str(g) == str(p))
    return correct, total


def numbered_only_accuracy(final, gt):
    """Stricter: denominator = tracklets with a real GT number; predicted -1 counts wrong."""
    correct = total = 0
    for tid, g in gt.items():
        if g == -1:
            continue
        p = final.get(tid, -1)
        total += 1
        correct += int(str(g) == str(p))
    return correct, total


oc, ot = overall_accuracy(final, gt)
nc, nt = numbered_only_accuracy(final, gt)

print(f"overall (SKIP_ILLEGIBLE both sides): {oc}/{ot} = {100.0 * oc / ot:.2f}%")
print(f"numbered-only (GT != -1 denom):      {nc}/{nt} = {100.0 * nc / nt:.2f}%")
print("paper claimed:                        87.45%")
print("local Windows repro baseline:         0.42%  (ViTPose swapped for KeypointRCNN)")


## 6. Outputs and cross-check

Everything lives under `/content/jersey-number-pipeline/out/SoccerNetResults/` on the
Colab VM (ephemeral — download before the runtime recycles):

- `final_results.json` — consolidated per-tracklet jersey number predictions (the file
  scored above).
- `jersey_id_results.json` — raw per-crop STR predictions before consolidation.
- `legible.json` / `illegible.json` — legibility-classifier split.
- `pose_results.json`, `crops/` — ViTPose keypoints and pose-guided crops.

To download the predictions for cross-check against the local laptop run:

```python
from google.colab import files
files.download("/content/jersey-number-pipeline/out/SoccerNetResults/final_results.json")
```

Compare against the local baseline's `final_results.json` (from
`~/jersey-number-pipeline/repro_soccernet.py` + `repro_str.py`) tracklet-by-tracklet to see
where the ViTPose-vs-KeypointRCNN pose swap and the missing ReID-outlier-filter stage
diverge from the authors' full pipeline.